Đọc dữ liệu

In [2]:

import os
import numpy as np
import pandas as pd

# Đường dẫn dữ liệu
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc dữ liệu
diamonds = pd.read_csv(file_path, sep=',')

print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Thông tin dữ liệu
diamonds.info()

print("\n--- 5 dòng đầu tiên ---")
print(diamonds.head())

print("\n--- Giá trị phân loại ---")
print(f"Cut: {diamonds['cut'].unique()}")
print(f"Color: {diamonds['color'].unique()}")
print(f"Clarity: {diamonds['clarity'].unique()}")

print("\n--- Thống kê mô tả ---")
print(diamonds.describe())


Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [3]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ cột không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ các dòng có kích thước không hợp lệ
initial_rows = len(diamonds)

diamonds = diamonds[
    (diamonds['x'] > 0) &
    (diamonds['y'] > 0) &
    (diamonds['z'] > 0)
].copy()

invalid_dimensions = initial_rows - len(diamonds)

print(
    f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu "
    "(x, y, z <= 0)."
)

# 3: Loại bỏ các dòng trùng lặp
num_duplicates = diamonds.duplicated().sum()

diamonds = diamonds.drop_duplicates()

print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa Ordinal
cut_categories = [
    'Fair', 'Good', 'Very Good', 'Premium', 'Ideal'
]

color_categories = [
    'J', 'I', 'H', 'G', 'F', 'E', 'D'
]

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 5: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng đầu sau tiền xử lý ---")
print(diamonds.head())



Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [4]:

from sklearn.model_selection import train_test_split

# X là các thuộc tính
X = diamonds.drop(columns=["price"])

# y là nhãn giá kim cương
y = diamonds["price"]

# Chia dữ liệu Train/Test theo tỷ lệ 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Chia thêm tập Validation từ Train
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print(
    f"Có {X_train.shape[0]} mẫu huấn luyện "
    f"và {X_test.shape[0]} mẫu kiểm tra"
)

print(f"Train dùng để tìm tham số: {X_train_sub.shape[0]} mẫu")
print(f"Validation: {X_val.shape[0]} mẫu")


Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra
Train dùng để tìm tham số: 34416 mẫu
Validation: 8604 mẫu


Training và tìm bộ tham số mức 4

In [6]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np
import pandas as pd
import time
from itertools import product

# Các giá trị tham số mức 4
n_estimators_values = [100, 150, 200]
max_depth_values = [15, 20, None]
min_samples_leaf_values = [1, 2]
min_samples_split_values = [2, 5]

# Tạo danh sách tổ hợp
param_combinations = list(product(
    n_estimators_values,
    max_depth_values,
    min_samples_leaf_values,
    min_samples_split_values
))

print(f"Tổng số tổ hợp: {len(param_combinations)}")

# Biến lưu kết quả
best_rmse = float('inf')
best_params = None
search_results = []

# Đo thời gian tìm kiếm
search_start = time.perf_counter()

for count, (n_estimators, max_depth,
            min_samples_leaf, min_samples_split) in enumerate(
                param_combinations, start=1):

    model = RandomForestRegressor(
        criterion="squared_error",
        random_state=42,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split
    )

    start = time.perf_counter()

    model.fit(X_train_sub, y_train_sub)

    training_time = time.perf_counter() - start

    # Đánh giá trên Validation
    y_pred_val = model.predict(X_val)

    rmse_val = np.sqrt(
        mean_squared_error(y_val, y_pred_val)
    )

    # Lưu kết quả từng tổ hợp
    search_results.append({
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "min_samples_split": min_samples_split,
        "RMSE Validation": rmse_val,
        "Training Time (s)": training_time
    })

    # Cập nhật bộ tham số tốt nhất
    if rmse_val < best_rmse:
        best_rmse = rmse_val
        best_params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split
        }

    print(
        f"Đã chạy {count}/{len(param_combinations)} tổ hợp - "
        f"RMSE Validation: {rmse_val:.4f}"
    )

search_time = time.perf_counter() - search_start

# Sắp xếp kết quả theo RMSE Validation
search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values(
    by="RMSE Validation"
)

print("\n5 BỘ THAM SỐ TỐT NHẤT")
print(
    search_results_df.head(5).to_string(index=False)
)

print("\nTham số tốt nhất:", best_params)
print(f"RMSE Validation tốt nhất: {best_rmse:.4f}")
print(f"Thời gian tìm kiếm: {search_time:.4f} giây")

# Huấn luyện lại trên toàn bộ Train
best_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=42,
    **best_params
)

start = time.perf_counter()

best_model.fit(X_train, y_train)

final_training_time = time.perf_counter() - start

# Dự đoán
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Đánh giá Train
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_pred_train)

# Đánh giá Test
mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_pred_test)

print("\nKẾT QUẢ RANDOM FOREST - HOLD OUT MỨC 4")

print(f"\nMAE Train: {mae_train:.4f}")
print(f"MSE Train: {mse_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Train: {r2_train:.4f}")

print(f"\nMAE Test: {mae_test:.4f}")
print(f"MSE Test: {mse_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")
print(f"R² Test: {r2_test:.4f}")

print(f"\nThời gian Train mô hình cuối: {final_training_time:.4f} giây")
print(f"Tổng thời gian tìm kiếm: {search_time:.4f} giây")


Tổng số tổ hợp: 36
Đã chạy 1/36 tổ hợp - RMSE Validation: 565.6290
Đã chạy 2/36 tổ hợp - RMSE Validation: 566.2040
Đã chạy 3/36 tổ hợp - RMSE Validation: 565.0538
Đã chạy 4/36 tổ hợp - RMSE Validation: 565.5596
Đã chạy 5/36 tổ hợp - RMSE Validation: 569.1610
Đã chạy 6/36 tổ hợp - RMSE Validation: 566.9765
Đã chạy 7/36 tổ hợp - RMSE Validation: 566.7028
Đã chạy 8/36 tổ hợp - RMSE Validation: 565.8679
Đã chạy 9/36 tổ hợp - RMSE Validation: 568.8984
Đã chạy 10/36 tổ hợp - RMSE Validation: 566.9030
Đã chạy 11/36 tổ hợp - RMSE Validation: 566.2987
Đã chạy 12/36 tổ hợp - RMSE Validation: 566.7577
Đã chạy 13/36 tổ hợp - RMSE Validation: 565.5382
Đã chạy 14/36 tổ hợp - RMSE Validation: 566.0286
Đã chạy 15/36 tổ hợp - RMSE Validation: 564.6077
Đã chạy 16/36 tổ hợp - RMSE Validation: 564.8481
Đã chạy 17/36 tổ hợp - RMSE Validation: 568.6506
Đã chạy 18/36 tổ hợp - RMSE Validation: 566.5654
Đã chạy 19/36 tổ hợp - RMSE Validation: 566.5687
Đã chạy 20/36 tổ hợp - RMSE Validation: 566.3442
Đã chạy 21

In [7]:

print("5 BỘ THAM SỐ TỐT NHẤT")
print(search_results_df.head(5).to_string(index=False))

print("\nTHAM SỐ TỐT NHẤT")
print(best_params)

print(f"\nRMSE Validation: {best_rmse:.4f}")

print("\nKẾT QUẢ RANDOM FOREST HOLD-OUT MỨC 4")

print(f"MAE Train: {mae_train:.4f}")
print(f"MSE Train: {mse_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Train: {r2_train:.4f}")

print(f"\nMAE Test: {mae_test:.4f}")
print(f"MSE Test: {mse_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")
print(f"R² Test: {r2_test:.4f}")

print(f"\nThời gian tìm kiếm: {search_time:.4f} giây")
print(f"Thời gian train cuối: {final_training_time:.4f} giây")


5 BỘ THAM SỐ TỐT NHẤT
 n_estimators  max_depth  min_samples_leaf  min_samples_split  RMSE Validation  Training Time (s)
          150       15.0                 2                  2       564.607746          12.654031
          150       15.0                 2                  5       564.848133          11.600966
          200       15.0                 2                  5       564.855506          17.002298
          200       15.0                 1                  2       565.016100          17.410902
          100       15.0                 2                  2       565.053827           7.431419

THAM SỐ TỐT NHẤT
{'n_estimators': 150, 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 2}

RMSE Validation: 564.6077

KẾT QUẢ RANDOM FOREST HOLD-OUT MỨC 4
MAE Train: 161.7726
MSE Train: 102527.2928
RMSE Train: 320.1988
R² Train: 0.9936

MAE Test: 266.6035
MSE Test: 270082.0118
RMSE Test: 519.6942
R² Test: 0.9829

Thời gian tìm kiếm: 631.4164 giây
Thời gian train cuối: 28.38